In [21]:
import pandas as pd
import pytz
import datetime as dt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
import joblib

In [22]:
def preprocess_QuantAQ(file_path, start_date, end_date):
    sensor = pd.read_csv(file_path, parse_dates=['timestamp'])
    sensor = sensor[['timestamp_local','rh','temp','no2']].dropna(subset=['no2'])
    sensor.rename(columns={'timestamp_local':'time'}, inplace=True)
    
    sensor['time'] = pd.to_datetime(sensor['time'])
    est = pytz.timezone('US/Eastern')
    sensor['time'] = sensor['time'].dt.tz_convert(est)
    sensor['time'] = sensor['time'].dt.strftime('%Y-%m-%d %H:%M:%S')
    sensor['time'] = pd.to_datetime(sensor['time'])
    sensor['day'] = sensor['time'].dt.date
    sensor['dayhour'] = sensor['time'].dt.strftime('%Y-%m-%d %H')
    sensor = sensor[(sensor['time'] >= start_date) & (sensor['time'] <= end_date)]

    return sensor


In [23]:
# returns a model that predicts daily no2 from hourly sensor data
def train_hourly_model(sensor_collocation_data, h_regular):
    temp = sensor_collocation_data.groupby('dayhour').agg(
        shourlyNO2=('no2', lambda x: x.mean(skipna=True)),
        shourlyRH=('rh', lambda x: x.mean(skipna=True)),
        shourlyTemp=('temp', lambda x: x.mean(skipna=True))
    ).reset_index()

    # inner join (default) - only keep rows that are in both dataframes
    collocation_data = pd.merge(h_regular, temp, on='dayhour').dropna().reset_index(drop=True)
    features = ['shourlyNO2', 'shourlyRH', 'shourlyTemp']
    target = collocation_data['rhourlyNO2']
    return LinearRegression().fit(collocation_data[features], target)

def train_daily_model(sensor_collocation_data, d_regular):
    temp = sensor_collocation_data.groupby('day').agg(
        sdailyNO2=('no2', lambda x: x.mean(skipna=True)),
        sdailyRH=('rh', lambda x: x.mean(skipna=True)),
        sdailyTemp=('temp', lambda x: x.mean(skipna=True))
    ).reset_index()

    # inner join (default) - only keep rows that are in both dataframes
    collocation_data = pd.merge(d_regular, temp, on='day').dropna().reset_index(drop=True)
    features = ['sdailyNO2', 'sdailyRH', 'sdailyTemp']
    target = collocation_data['rdailyNO2']
    return LinearRegression().fit(collocation_data[features], target)



In [24]:
quantAQ00589 = preprocess_QuantAQ("../ShortenedData/MOD-00589-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00590 = preprocess_QuantAQ("../ShortenedData/MOD-00590-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00591 = preprocess_QuantAQ("../ShortenedData/MOD-00591-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00592 = preprocess_QuantAQ("../ShortenedData/MOD-00592-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00593 = preprocess_QuantAQ("../ShortenedData/MOD-00593-RAW.csv", '2025-05-31', '2025-07-14')
quantAQ00595 = preprocess_QuantAQ("../ShortenedData/MOD-00595-RAW.csv", '2025-05-31', '2025-07-14')

In [25]:
# load and clean GAPA data
gapa = pd.read_csv('data/SouthDeKalb_NO2CAPS_May31-July16.csv', header=2)
gapa.columns = ['Date', 'NO2 CAPS']

gapa['time'] = pd.to_datetime(gapa['Date'], format='%d-%b-%Y %H:%M')
gapa['day'] = gapa['time'].dt.strftime("%Y-%m-%d")
gapa['day'] = pd.to_datetime(gapa['day']).dt.date
gapa['dayhour'] = gapa['time'].dt.strftime("%Y-%m-%d %H")
gapa.rename(columns={'NO2 CAPS':'rno2'}, inplace=True)

gapa = gapa.sort_values('time').reset_index(drop=True).dropna(subset=['rno2'])

h_gapa = gapa.groupby('dayhour').agg(
    rhourlyNO2=('rno2', lambda x: x.mean(skipna=True)),
).reset_index()

d_gapa = gapa.groupby('day').agg(
    rdailyNO2=('rno2', lambda x: pd.to_numeric(x, errors='coerce').mean(skipna=True)),
).reset_index()




In [26]:
hmodel_00589 = train_hourly_model(quantAQ00589, h_gapa)
dmodel_00589 = train_daily_model(quantAQ00589, d_gapa)
hmodel_00590 = train_hourly_model(quantAQ00590, h_gapa)
dmodel_00590 = train_daily_model(quantAQ00590, d_gapa)
hmodel_00591 = train_hourly_model(quantAQ00591, h_gapa)
dmodel_00591 = train_daily_model(quantAQ00591, d_gapa)
hmodel_00592 = train_hourly_model(quantAQ00592, h_gapa)
dmodel_00592 = train_daily_model(quantAQ00592, d_gapa)
hmodel_00593 = train_hourly_model(quantAQ00593, h_gapa)
dmodel_00593 = train_daily_model(quantAQ00593, d_gapa)
hmodel_00595 = train_hourly_model(quantAQ00595, h_gapa)
dmodel_00595 = train_daily_model(quantAQ00595, d_gapa)

In [27]:
# save models
joblib.dump(hmodel_00589, 'models/hmodel_00589.joblib')
joblib.dump(dmodel_00589, 'models/dmodel_00589.joblib')
joblib.dump(hmodel_00590, 'models/hmodel_00590.joblib')
joblib.dump(dmodel_00590, 'models/dmodel_00590.joblib')
joblib.dump(hmodel_00591, 'models/hmodel_00591.joblib')
joblib.dump(dmodel_00591, 'models/dmodel_00591.joblib')
joblib.dump(hmodel_00592, 'models/hmodel_00592.joblib')
joblib.dump(dmodel_00592, 'models/dmodel_00592.joblib')
joblib.dump(hmodel_00593, 'models/hmodel_00593.joblib')
joblib.dump(dmodel_00593, 'models/dmodel_00593.joblib')
joblib.dump(hmodel_00595, 'models/hmodel_00595.joblib')
joblib.dump(dmodel_00595, 'models/dmodel_00595.joblib')


['models/dmodel_00595.joblib']